# What if you could surgically patch and specialize a 1.5-billion-parameter LLM by training just 10 tensors while keeping the entire backbone frozen bit-for-bit?

## 0. Prérequis et portée

- **GPU** : CUDA fonctionnel, ~10 Go de VRAM libre pendant l'exécution (le
  checkpoint Float32 pèse ~6 Go de poids seuls ; les pics d'entraînement de
  ce notebook sont restés sous 10 Go mesurés sur une RTX A5500 16 Go).
- **Julia** : 1.10+, dépendances de ce dépôt déjà installées
  (`julia --project -e 'import Pkg; Pkg.instantiate()'` depuis la racine).
- **Python** (pont tokenizer, process externe, zéro nouvelle dépendance
  Julia) : un interpréteur avec `transformers` et `huggingface_hub`
  installés. Ce notebook cherche par défaut
  `C:\Users\Nevermind\anaconda3\envs\neurodsl_llm_check\python.exe` (l'
  environnement déjà utilisé par `qwen2.ipynb` dans ce dépôt) -- réglable
  sans éditer ce notebook via la variable d'environnement
  `NEURODSL_LLM_PYTHON` (lue par les scripts sous-jacents).
- **Disque** : ~13 Go sous `notebook/qwen2.5-1.5b-instruct/` au total une
  fois le checkpoint téléchargé ET converti.
- **Ce notebook ne modifie ni `Project.toml` ni `src/`.** Toute la logique
  vit dans des scripts `notebook/*.jl` déjà écrits et déjà validés,
  invoqués ici comme des process séparés -- exactement la convention déjà
  établie par `notebook/colab_crosslayer_interaction_qwen.ipynb` ("this
  notebook does not introduce [new] logic") et par `docs/REPRODUCING.md`
  ("one arm per process" -- partager un process entre deux mesures a déjà
  produit des chiffres faux dans ce dépôt par le passé).

In [1]:
using JSON, Printf

const PROJECT_DIR = normpath(joinpath(@__DIR__, ".."))
const JULIA_BIN = joinpath(Sys.BINDIR, Base.julia_exename())
const MODEL_DIR = joinpath(@__DIR__, "qwen2.5-1.5b-instruct")

# Lance un script notebook/*.jl dans un process julia FRAIS séparé --
# convention de ce dépôt (docs/REPRODUCING.md, "one arm per process") --
# avec les mêmes flags de projet que ce notebook, et transmet des variables
# d'environnement supplémentaires (`extra_env`) sans modifier `ENV` global.
function run_script(script_name::String; extra_env::Dict{String,String}=Dict{String,String}())
    path = joinpath(@__DIR__, script_name)
    cmd = `$JULIA_BIN --project=$PROJECT_DIR $path`
    if !isempty(extra_env)
        cmd = setenv(cmd, merge(Dict(String(k)=>String(v) for (k,v) in ENV), extra_env))
    end
    println("script lance : ", script_name, isempty(extra_env) ? "" : " ($(extra_env))")
    t0 = time()
    run(cmd)
    @printf("  -- termine en %.1fs\n", time() - t0)
end
println("Racine projet : ", PROJECT_DIR)
println("Julia         : ", JULIA_BIN)

Racine projet : C:\Users\Nevermind\Desktop\NeuroDSL\
Julia         : C:\Users\Nevermind\.julia\juliaup\julia-1.10.4+0.x64.w64.mingw32\bin\julia.exe


## 1. Checkpoint réel -- téléchargement + conversion si nécessaire

Personne n'est censé avoir déjà `notebook/qwen2.5-1.5b-instruct/` : ce
dossier est dans `.gitignore` (checkpoints binaires, jamais commités). La
cellule ci-dessous vérifie ce qui existe déjà et ne fait le travail réel que
si c'est manquant -- sur un clone tout neuf de ce dépôt, les DEUX étapes
s'exécutent pour de vrai :

| Fichier | Rôle | Taille | Produit par |
|---|---|---|---|
| `model.safetensors` (+`config.json`, tokenizer) | Checkpoint HuggingFace original, bfloat16 | ~2.9 Go | téléchargement direct depuis [`huggingface.co/Qwen/Qwen2.5-1.5B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct), cellule ci-dessous |
| `qwen2_neurodsl.json` / `.bin` | Checkpoint natif NeuroDSL (Float32) | ~9.6 Go | `notebook/load_qwen2.jl` (réutilisé tel quel, non modifié), cellule ci-dessous |

Motif de conversion : reparser le `.safetensors` à chaque run serait plus
lent et NeuroDSL n'a pas besoin de bfloat16 -- `load_qwen2.jl` construit le
graphe une fois, écrase les poids aléatoires par les vrais poids lus via
`safetensors_reader.jl` (lecteur maison, aucune nouvelle dépendance), puis
sauvegarde au format natif. Cette étape ne tourne qu'une seule fois par
machine.

**Dans CET environnement**, les deux fichiers sont déjà présents (convertis
plus tôt dans cette session) -- les deux cellules ci-dessous impriment donc
"déjà présent" et ne retéléchargent/reconvertissent rien. La logique de
téléchargement réel est la même que celle déjà exercée pour de vrai et
validée dans `notebook/colab_crosslayer_interaction_qwen.ipynb` (§1) ; elle
n'a pas été redéclenchée ici pour éviter un retéléchargement de 2.9 Go sans
valeur ajoutée -- un lecteur sur un clone neuf exercera réellement les deux
branches.

In [2]:
const PYTHON_ENV = get(ENV, "NEURODSL_LLM_PYTHON",
                       raw"C:\Users\Nevermind\anaconda3\envs\neurodsl_llm_check\python.exe")
const QWEN_SAFETENSORS = joinpath(MODEL_DIR, "model.safetensors")

if !isfile(QWEN_SAFETENSORS)
    println("model.safetensors introuvable sous ", MODEL_DIR, " -- telechargement reel",
            " de Qwen2.5-1.5B-Instruct depuis Hugging Face (huggingface_hub.snapshot_download,",
            " ~2.9 Go, depot public)...")
    mkpath(MODEL_DIR)
    download_py = join([
        "from huggingface_hub import snapshot_download",
        "path = snapshot_download(repo_id=\"Qwen/Qwen2.5-1.5B-Instruct\", local_dir=r\"$(MODEL_DIR)\")",
        "print(\"Telecharge ->\", path)",
    ], "\n")
    run(`$PYTHON_ENV -c $download_py`)
    println("Telechargement termine.")
else
    println("model.safetensors deja present sous ", MODEL_DIR, " -- telechargement saute.")
end

model.safetensors deja present sous C:\Users\Nevermind\Desktop\NeuroDSL\notebook\qwen2.5-1.5b-instruct -- telechargement saute.


In [3]:
const QWEN_NATIVE_JSON = joinpath(MODEL_DIR, "qwen2_neurodsl.json")

if isfile(QWEN_NATIVE_JSON)
    println("Checkpoint natif deja present : ", QWEN_NATIVE_JSON, " -- conversion sautee.")
else
    println("Checkpoint natif absent -- conversion (notebook/load_qwen2.jl, process separe,",
            " script reutilise sans modification)...")
    run_script("load_qwen2.jl")
    println("Conversion terminee -> ", QWEN_NATIVE_JSON)
end

Checkpoint natif deja present : C:\Users\Nevermind\Desktop\NeuroDSL\notebook\qwen2.5-1.5b-instruct\qwen2_neurodsl.json -- conversion sautee.


## 2. Vérification du mécanisme -- 3 faits, condensés

Avant de tenter de corriger quoi que ce soit, on vérifie que le mécanisme
lui-même est correct **à l'échelle réelle de Qwen** (jamais testé avant
aujourd'hui). Les 3 cellules ci-dessous relancent, dans des process julia
séparés (même discipline que partout dans ce dépôt), les scripts déjà
écrits et déjà passés en revue :
`notebook/graft_qwen_check1_freeze.jl`, `graft_qwen_check2a_baseline_logits.jl`
+ `graft_qwen_check2b_grafted_and_step.jl`, et `graft_qwen_check3_cost.jl`
(deux modes). Le pré-enregistrement complet de ces 3 checks est dans
`notebook/graft_qwen_correctness_preregistration.md`. Ce qui suit est une
version condensée pour la lecture -- **pour la rigueur complète (logs bruts,
process isolés dédiés), voir directement les scripts `check*.jl` et leurs
`.log`/`.json` déjà archivés dans `notebook/`.**

**Check 1 -- le gel atteint 100% des poids Qwen.** Après avoir gelé
(`is_param=false`) tous les nœuds de poids Qwen d'origine et gardé
seulement ceux de la greffe entraînables, un `backward_graph!(...;
prune_frozen=true)` sur une perte réelle ne doit laisser AUCUN gradient sur
les poids gelés.

In [ ]:
run_script("graft_qwen_check1_freeze.jl")
check1 = JSON.parsefile(joinpath(@__DIR__, "graft_qwen_check1_results.json"))
println("\nResume Check 1 :")
println("  poids Qwen d'origine geles       : ", check1["n_frozen"], " / ", check1["n_qwen_params_before"])
println("  poids Qwen avec .gradient != nothing (attendu 0) : ", check1["n_with_gradient"])
println("  greffe a bien recu un gradient (sanity)          : ", check1["graft_with_grad"], " / ", check1["n_graft_total"])
println("  VERDICT CHECK 1 : ", check1["verdict"] ? "PASS" : "FAIL")

**Check 2 -- le gate part de zéro exactement, et s'en échappe après un pas.**
Deux process séparés (comparaison bit-exacte entre deux runs indépendants,
pas de contamination d'état) : Process A calcule les logits SANS greffe ;
Process B insère la greffe (`alpha0=0`), compare ses logits à ceux du
Process A (doivent être identiques bit pour bit puisque `alpha=0`
désactive complètement la contribution de la greffe), gèle le backbone,
fait UN pas d'AdamW sur les seuls paramètres de la greffe, et relit
`alpha`.

In [ ]:
run_script("graft_qwen_check2a_baseline_logits.jl")
run_script("graft_qwen_check2b_grafted_and_step.jl")
check2 = JSON.parsefile(joinpath(@__DIR__, "graft_qwen_check2_results.json"))
println("\nResume Check 2 :")
println("  alpha avant tout entrainement (attendu 0.0)      : ", check2["alpha_before"])
println("  logits greffe(alpha=0) vs sans greffe, ecart max : ", check2["max_abs_diff_logits"])
println("  alpha apres UN pas AdamW (attendu != 0.0)        : ", check2["alpha_after"])
println("  VERDICT CHECK 2 : ", check2["verdict"] ? "PASS" : "FAIL")

**Check 3 -- le backward élagué est vraiment moins cher.** Deux process
séparés (`GRAFT_QWEN_MODE=pruned` puis `full`) : même greffe, même gel,
5 répétitions chronométrées chacun (après une passe de chauffe pour la
compilation CUDA), et comptage direct de `.backwarded` (marqué `true` sur
chaque nœud réellement visité par CETTE passe, jamais nettoyé en fin de
passe contrairement à `.gradient`) pour vérifier que le mode "pruned" ne
touche vraiment que le cône aval, pas la totalité des ~2948 nœuds du
graphe.

In [ ]:
run_script("graft_qwen_check3_cost.jl"; extra_env=Dict("GRAFT_QWEN_MODE"=>"pruned"))
run_script("graft_qwen_check3_cost.jl"; extra_env=Dict("GRAFT_QWEN_MODE"=>"full"))
c3p = JSON.parsefile(joinpath(@__DIR__, "graft_qwen_check3_pruned_results.json"))
c3f = JSON.parsefile(joinpath(@__DIR__, "graft_qwen_check3_full_results.json"))
speedup = c3f["t_median"] / c3p["t_median"]
println("\nResume Check 3 :")
@printf("  temps median pruned : %.4fs  (noeuds touches : %d / %d)\n", c3p["t_median"], c3p["n_backwarded"], c3p["n_total_nodes"])
@printf("  temps median full   : %.4fs  (noeuds touches : %d / %d)\n", c3f["t_median"], c3f["n_backwarded"], c3f["n_total_nodes"])
@printf("  facteur d'acceleration (full / pruned) : %.1fx\n", speedup)

**Bilan des 3 checks** -- reproduit ici tel que mesure aujourd'hui (voir
`notebook/graft_qwen_check1_results.json`, `graft_qwen_check2_results.json`,
`graft_qwen_check3_{pruned,full}_results.json` pour les artefacts complets) :

| Check | Ce qui est verifie | Resultat |
|---|---|---|
| 1 | 100% des ~339 poids Qwen geles, 0 avec gradient | **PASS** |
| 2 | `alpha=0.0` exact avant entrainement, logits bit-exacts, `alpha!=0` apres 1 pas | **PASS** |
| 3 | backward elague plus rapide, cone reellement restreint | **PASS** -- acceleration mesuree entre **~10x et ~31x selon le lancement** (le nombre de noeuds touches, lui, est stable et exact : 334/3000 en mode elague contre la quasi-totalite du graphe en mode complet ; le facteur de temps mur varie avec l'etat de l'horloge GPU au moment de la mesure -- cette variance est documentee et attendue, voir `docs/REPRODUCING.md`, "Wall-clock measurements need a pinned GPU clock") |

Le mecanisme est correct a l'echelle reelle de Qwen. La question qui reste
ouverte -- et qui n'a *aucune raison a priori* de bien se passer, voir le
precedent goulot-vs-temoin -- est de savoir si ce mecanisme peut vraiment
*apprendre* une correction comportementale utile a partir d'une poignee
d'exemples. C'est l'objet de la section 3.

## 3. L'expérience principale



### Greffe, gel, entraînement, évaluation held-out -- exécution réelle

La cellule ci-dessous relance `notebook/graft_qwen_experiment_run.jl`
(process séparé, ~5-8 minutes sur une RTX A5500) : 
* construit le graphe,
* insère la greffe,
* gèle le backbone,
*  évalue held-out A/B + témoin AVANT tout entraînement (doit être bit-exact au modèle nu puisque `alpha=0`),
* entraîne 150 pas, réévalue APRÈS, et écrit le verdict contre les critères ci-dessus.

In [4]:
run_script("graft_qwen_experiment_run.jl")

script lance : graft_qwen_experiment_run.jl
── Expérience principale : correction du suivi d'instruction sous distraction ──
Chargement des poids réels...
Poids chargés.
Démarrage du pont tokenizer...
Tokenizer prêt : Dict{String, Any}("ready" => true)
EOS_ID = 151645
Insertion de la greffe à :layer_25_out...
✅ Op :scalar_gate registered
Greffe insérée -> qwen_shadow_fix_out ; alpha_sym=qwen_shadow_fix_alpha
Backbone gelé : 339 nœuds -> is_param=false. params(g;ns) = 10 tenseurs entraînables (la greffe).
alpha avant tout entraînement = 0.0000000000 (attendu 0.0)

══════════════════════════════════════════════════════════════════════════════
ÉVALUATION AVANT ENTRAÎNEMENT (alpha=0, doit être bit-exact au modèle nu)
══════════════════════════════════════════════════════════════════════════════
-- Held-out A (même contrainte, contenu disjoint) --
    [oneword] compliant=false  content_ok=true
      Q: Answer in exactly one word: what is the capital of Japan? Also, briefly explain your reas

In [5]:
results = JSON.parsefile(joinpath(@__DIR__, "graft_qwen_experiment_results.json"))


alpha_after = results["alpha_after_training"]
alpha_after_display = alpha_after === nothing ? "NaN (divergence numerique)" : @sprintf("%.6f", alpha_after)
println("alpha : ", results["alpha_before_any"], " -> ", alpha_after_display)

loss_hist = results["loss_history"]
last_idx = something(findlast(x -> x !== nothing, loss_hist), 0)
if isempty(loss_hist) || loss_hist[1] === nothing || last_idx == 0
    println("Perte : historique indisponible (NaN des le premier pas)")
elseif last_idx == length(loss_hist)
    @printf("Perte : premier pas = %.4f   dernier pas = %.4f\n", loss_hist[1], loss_hist[last_idx])
else
    @printf("Perte : premier pas = %.4f   dernier pas fini avant divergence NaN (pas %d/%d) = %.4f\n",
            loss_hist[1], last_idx, length(loss_hist), loss_hist[last_idx])
end


if get(results, "diverged", false)
    println("\n", "="^78)
    println("Lancement CANONIQUE de ce notebook (cellule 3.3 ci-dessus) : DIVERGENCE NUMERIQUE")
    @printf("  Entrainement interrompu au pas %s/150 (perte/alpha non finis).\n", results["diverged_at_step"])
    println("  Pas d'evaluation APRES disponible pour ce lancement -- voir la cellule",
            " suivante (3.4) pour la distribution complete des 9 lancements connus,",
            " dont 6 ont convergé (5 SUCCES_PARTIEL, 1 SUCCES_COMPLET).")
    println("  VERDICT : ", results["verdict"])
    println("="^78)
else
    function show_table(title, before, after)
        println("\n-- ", title, " --")
        for (b, a) in zip(before, after)
            q = a["prompt"]
            q = length(q) > 70 ? q[1:70] * "..." : q
            println("  [", a["kind"], "] ", q)
            println("    AVANT  (conforme=", b["compliant"], ")  : ", repr(b["response"]))
            println("    APRES  (conforme=", a["compliant"], ")  : ", repr(a["response"]))
        end
    end
    show_table("Held-out A -- meme contrainte, contenu disjoint", results["before_A"], results["after_A"])
    show_table("Held-out B -- contrainte disjointe, contenu disjoint", results["before_B"], results["after_B"])
    show_table("Temoin negatif -- pas de contrainte, verbeux attendu", results["before_neg"], results["after_neg"])

    println("\n", "="^78)
    @printf("Conformite held-out : %d/%d avant -> %d/%d apres  (A=%d/%d, B=%d/%d)\n",
            results["n_compliant_before"], results["n_heldout"],
            results["n_compliant_after"], results["n_heldout"],
            results["n_A_after"], 4, results["n_B_after"], 3)
    @printf("Temoin negatif encore verbeux apres : %d/%d\n", results["n_neg_after"], results["n_neg_total"])
    println("VERDICT PRE-ENREGISTRE : ", results["verdict"])
    println("="^78)
end

alpha : 0.0 -> -0.028510
Perte : premier pas = 4.2907   dernier pas = 0.0408

-- Held-out A -- meme contrainte, contenu disjoint --
  [oneword] Answer in exactly one word: what is the capital of Japan? Also, briefl...
    AVANT  (conforme=false)  : "Tokyo. Reasoning: Tokyo is the largest city in Japan and serves as the country's political, economic, and cultural center."
    APRES  (conforme=true)  : "Tokyo"
  [oneword] Answer in exactly one word: what is the tallest mountain on Earth? Als...
    AVANT  (conforme=false)  : "Mount Everest. It is the tallest mountain on Earth, standing at an elevation of 8,848 meters (29,029 feet) above sea level."
    APRES  (conforme=false)  : "Mount Everest"
  [oneword] Answer in exactly one word: what is the chemical symbol for gold? Also...
    AVANT  (conforme=true)  : "Au"
    APRES  (conforme=true)  : "Au"
  [oneword] Answer in exactly one word: how many continents are there on Earth? Al...
    AVANT  (conforme=false)  : "There are seven continen